<a href="https://colab.research.google.com/github/kmeza3278/etl-data-pipeline-17-3278-2021-/blob/main/Parcia_Fenvios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# ETL - F_envios.csv
# Ajustado al archivo real
# ============================================================

import pandas as pd
import re

# ------------------------------------------------------------
# 1. URL
# ------------------------------------------------------------
url_envios = "https://raw.githubusercontent.com/kmeza3278/etl-data-pipeline-17-3278-2021-/refs/heads/main/data/F_envios.csv"

# ------------------------------------------------------------
# 2. CARGA
# ------------------------------------------------------------
envios_raw = pd.read_csv(url_envios)

print("Archivo cargado")
print("Dimensiones:", envios_raw.shape)
display(envios_raw.head())

# ------------------------------------------------------------
# 3. COPIA
# ------------------------------------------------------------
envios = envios_raw.copy()

# ------------------------------------------------------------
# 4. FUNCIONES BASE
# ------------------------------------------------------------
def normalizar_columnas(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    return df

def limpiar_texto(df):
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace(["nan", "None", "NULL", "null", ""], pd.NA)
    return df

def convertir_fecha(col):
    return pd.to_datetime(col, errors="coerce")

def normalizar_estado_envio(valor):
    if pd.isna(valor):
        return pd.NA

    valor = str(valor).strip().lower()

    mapa = {
        "en ruta": "En Ruta",
        "entregado": "Entregado",
        "devuelto": "Devuelto",
        "preparando": "Preparando"
    }

    return mapa.get(valor, valor.title())

def id_envio_valido(valor):
    if pd.isna(valor):
        return False
    return bool(re.fullmatch(r"ENV\d+", str(valor).strip().upper()))

def id_pedido_valido(valor):
    if pd.isna(valor):
        return False
    return bool(re.fullmatch(r"PED\d+", str(valor).strip().upper()))

def motivo_rechazo(row):
    motivos = []

    if pd.isna(row["id_envio"]):
        motivos.append("id_envio_vacio")
    elif not id_envio_valido(row["id_envio"]):
        motivos.append("id_envio_invalido")

    if pd.isna(row["id_pedido"]):
        motivos.append("id_pedido_vacio")
    elif not id_pedido_valido(row["id_pedido"]):
        motivos.append("id_pedido_invalido")

    if pd.isna(row["direccion"]):
        motivos.append("direccion_vacia")

    if pd.isna(row["fecha_envio"]):
        motivos.append("fecha_envio_invalida")

    if pd.isna(row["estado_envio"]):
        motivos.append("estado_envio_vacio")

    return ",".join(motivos)

# ------------------------------------------------------------
# 5. LIMPIEZA BASE
# ------------------------------------------------------------
envios = normalizar_columnas(envios)
envios = limpiar_texto(envios)
envios = envios.drop_duplicates()

print("\nColumnas reales:", list(envios.columns))

# ------------------------------------------------------------
# 6. LIMPIEZA ESPECIFICA
# ------------------------------------------------------------
envios["id_envio"] = envios["id_envio"].astype("string").str.strip().str.upper()
envios["id_pedido"] = envios["id_pedido"].astype("string").str.strip().str.upper()
envios["direccion"] = envios["direccion"].astype("string").str.strip()
envios["fecha_envio"] = convertir_fecha(envios["fecha_envio"])
envios["estado_envio"] = envios["estado_envio"].apply(normalizar_estado_envio)

# reemplazar strings vacíos por NA después de la limpieza
for col in ["id_envio", "id_pedido", "direccion", "estado_envio"]:
    envios[col] = envios[col].replace(["", "nan", "None", "<NA>"], pd.NA)

# ------------------------------------------------------------
# 7. VALIDACION
# ------------------------------------------------------------
condicion_valida = (
    envios["id_envio"].notna() &
    envios["id_pedido"].notna() &
    envios["direccion"].notna() &
    envios["fecha_envio"].notna() &
    envios["estado_envio"].notna() &
    envios["id_envio"].apply(id_envio_valido) &
    envios["id_pedido"].apply(id_pedido_valido)
)

envios_curated = envios[condicion_valida].copy()
envios_rejects = envios[~condicion_valida].copy()

envios_rejects["motivo_rechazo"] = envios_rejects.apply(motivo_rechazo, axis=1)

# ------------------------------------------------------------
# 8. RESULTADOS
# ------------------------------------------------------------
print("\n=== RESUMEN ===")
print("Total procesados:", len(envios))
print("Curated:", len(envios_curated))
print("Rejects:", len(envios_rejects))

print("\n=== ESTADOS NORMALIZADOS ===")
display(envios["estado_envio"].value_counts(dropna=False))

print("\n=== MUESTRA CURATED ===")
display(envios_curated.head())

print("\n=== MUESTRA REJECTS ===")
display(envios_rejects.head())

# ------------------------------------------------------------
# 9. EXPORTAR
# ------------------------------------------------------------
envios_curated.to_csv("envios_curated.csv", index=False)
envios_rejects.to_csv("envios_rejects.csv", index=False)

print("\nArchivos generados:")
print("- envios_curated.csv")
print("- envios_rejects.csv")


from google.colab import files

files.download("envios_curated.csv")
files.download("envios_rejects.csv")

Archivo cargado
Dimensiones: (216, 5)


,id_envio,id_pedido,direccion,fecha_envio,estado_envio
0,ENV8000,PED7100,"Boulevard Constitución, San Salvador",2024-06-02,en ruta
1,ENV8001,PED7222,"Calle El Mirador, La Libertad",2024-05-07,entregado
2,ENV8002,PED7191,"Calle El Mirador, Sonsonate",2024-05-30,devuelto
3,ENV8003,PED7186,"Av. Roosevelt, San Miguel",2024-07-03,en ruta
4,ENV8004,PED7043,"Calle Principal, San Salvador",2024-01-09,preparando



Columnas reales: ['id_envio', 'id_pedido', 'direccion', 'fecha_envio', 'estado_envio']

=== RESUMEN ===
Total procesados: 210
Curated: 185
Rejects: 25

=== ESTADOS NORMALIZADOS ===


,count
estado_envio,
Entregado,65
Devuelto,55
En Ruta,46
Preparando,44



=== MUESTRA CURATED ===


...


Archivos generados:
- envios_curated.csv
- envios_rejects.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>